# ***Detection Model Dataset Making***

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
BASE_PATH = "/content/drive/MyDrive/physalis_project/raw_datasets"

In [ ]:
import os

datasets = [
    "01_physalis_peruviana_roboflow",
    "02_physalis_madurez",
    "03_physalis_hojas_frutas",
    "04_aguaymanto_daniel",
    "06_iphone_field_photos"
]

BASE_PATH = "/content/drive/MyDrive/physalis_project/raw_datasets"

def clean_labels(dataset_path):
    splits = ["train", "valid", "test"]

    total_files = 0
    fixed_labels = 0
    removed_labels = 0

    for split in splits:
        labels_path = os.path.join(dataset_path, split, "labels")

        if not os.path.exists(labels_path):
            continue

        for file in os.listdir(labels_path):
            if file.endswith(".txt"):
                total_files += 1
                file_path = os.path.join(labels_path, file)

                with open(file_path, "r") as f:
                    lines = f.readlines()

                new_lines = []

                for line in lines:
                    parts = line.strip().split()

                    if len(parts) != 5:
                        removed_labels += 1
                        continue

                    try:
                        vals = list(map(float, parts))
                        vals[0] = 0  # FORCE CLASS = 0

                        if all(0 <= v <= 1 for v in vals[1:]):
                            new_lines.append(
                                " ".join([str(vals[0])] + [str(v) for v in vals[1:]]) + "\n"
                            )
                            fixed_labels += 1
                        else:
                            removed_labels += 1

                    except:
                        removed_labels += 1
                        continue

                with open(file_path, "w") as f:
                    f.writelines(new_lines)

    print(f"\n📂 Dataset: {os.path.basename(dataset_path)}")
    print(f"Files processed: {total_files}")
    print(f"Labels fixed: {fixed_labels}")
    print(f"Labels removed: {removed_labels}")


for d in datasets:
    clean_labels(os.path.join(BASE_PATH, d))

print("\n🔥 DATA CLEANED PERFECTLY")


📂 Dataset: 01_physalis_peruviana_roboflow
Files processed: 6314
Labels fixed: 2552
Labels removed: 0

📂 Dataset: 02_physalis_madurez
Files processed: 1263
Labels fixed: 1807
Labels removed: 0

📂 Dataset: 03_physalis_hojas_frutas
Files processed: 814
Labels fixed: 1167
Labels removed: 0

📂 Dataset: 04_aguaymanto_daniel
Files processed: 522
Labels fixed: 1193
Labels removed: 0

📂 Dataset: 06_iphone_field_photos
Files processed: 1013
Labels fixed: 21534
Labels removed: 0

🔥 DATA CLEANED PERFECTLY


In [ ]:
import os
import shutil
datasets = [
    "01_physalis_peruviana_roboflow",
    "02_physalis_madurez",
    "03_physalis_hojas_frutas",
    "04_aguaymanto_daniel",
    "06_iphone_field_photos"
]
BASE_PATH = "/content/drive/MyDrive/physalis_project/raw_datasets"
MERGED_PATH = "/content/drive/MyDrive/physalis_project/merged_dataset"
os.makedirs(MERGED_PATH + "/images", exist_ok=True)
os.makedirs(MERGED_PATH + "/labels", exist_ok=True)
count = 0
missing = 0
for dataset in datasets:
    dataset_path = os.path.join(BASE_PATH, dataset)
    for root, dirs, files in os.walk(dataset_path):
        if "images" in root:
            for file in files:
                if file.endswith((".jpg", ".png", ".jpeg")):
                    img_src = os.path.join(root, file)
                    label_file = file.replace(".jpg", ".txt").replace(".png", ".txt").replace(".jpeg", ".txt")
                    lbl_src = root.replace("images", "labels") + "/" + label_file
                    if os.path.exists(lbl_src):
                        new_name = dataset + "_" + file
                        shutil.copy(img_src, os.path.join(MERGED_PATH, "images", new_name))
                        shutil.copy(lbl_src, os.path.join(MERGED_PATH, "labels", new_name.replace(".jpg", ".txt")))
                        count += 1
                    else:
                        missing += 1
print("Merged Data Cleaned")
print("Total valid samples:", count)
print("Missing labels skipped:", missing)

Merged Data Cleaned
Total valid samples: 9926
Missing labels skipped: 0


In [ ]:
import os
import shutil
import random
MERGED_PATH = "/content/drive/MyDrive/physalis_project/merged_dataset"
FINAL_PATH = "/content/drive/MyDrive/physalis_project/final_dataset"
splits = ["train", "valid", "test"]
for split in splits:
    os.makedirs(f"{FINAL_PATH}/{split}/images", exist_ok=True)
    os.makedirs(f"{FINAL_PATH}/{split}/labels", exist_ok=True)
images = os.listdir(MERGED_PATH + "/images")
random.shuffle(images)
total = len(images)
train_end = int(0.8 * total)
valid_end = int(0.9 * total)
train = images[:train_end]
valid = images[train_end:valid_end]
test = images[valid_end:]
def move_files(file_list, split):
    for file in file_list:
        img_src = os.path.join(MERGED_PATH, "images", file)
        lbl_src = os.path.join(MERGED_PATH, "labels", file.replace(".jpg", ".txt"))
        shutil.copy(img_src, f"{FINAL_PATH}/{split}/images/{file}")
        shutil.copy(lbl_src, f"{FINAL_PATH}/{split}/labels/{file.replace('.jpg', '.txt')}")
move_files(train, "train")
move_files(valid, "valid")
move_files(test, "test")
print("FINAL DATASET READY (80/10/10)")

FINAL DATASET READY (80/10/10)


In [ ]:
!ls /content/drive/MyDrive/physalis_project

final_dataset  merged_dataset  Notebooks  raw_datasets


In [ ]:
!ls /content/drive/MyDrive/physalis_project/final_dataset/train
!ls /content/drive/MyDrive/physalis_project/final_dataset/valid
!ls /content/drive/MyDrive/physalis_project/final_dataset/test

images	labels
images	labels
images	labels
